# TPCのドリフト速度：rawツリー → 飛跡積算図 → クロック分布 → 較正ファイル

`RawTreeWriter_sm3` のヒットからアノード・カソードの飛跡を積算し、ドリフト領域の両端を求めます。
既存の `scripts/calib/make_tpc_drift_velocity.py` と同じ計算法を使います。

$$v = \frac{L-z_{\rm GEM}}{t_{\rm top}-t_{\rm GEM}},\qquad
z = v(t-t_{\rm GEM})+z_{\rm GEM}.$$

SMILE-3の寸法は μ-PIC〜GEM が10 mm、μ-PIC〜天板が317 mmで、両端間は307 mmです。
速度の単位は **mm/clock**。mm/µsへの換算にはTPCエンコーダーのクロック周期の確認が必要です。
ドリフト体積にほぼ一様に電離がある測定が必要です。箱型でない分布の端を無理に較正値にしません。

In [ ]:
import sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import Image, display
import ROOT
ROOT.gROOT.SetBatch(True)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_script/run_chain.py').exists())
SCRIPTS = REPO / 'scripts/calib'
sys.path.insert(0, str(SCRIPTS))
CACHE = REPO / 'tutorials/cache'
CACHE.mkdir(exist_ok=True)
PYTHON = '/usr/bin/python3'  # numpy / PyROOT のある Python
TREE = 'RawTreeWriter_sm3/raw'

## 1. 入力ROOTファイル

このノートブックは **既存の `RawTreeWriter_sm3/raw` を読む**もので、生データを自動で再解析しません。
下の `RAWS` を実際のROOTファイルに変更します。複数ファイルは同じ測定条件のものを指定してください。
run70 は ETCC、run71〜73 は TPC_cal です。測定条件と線源を確認し、モードだけで較正に適しているとは判断しません。
前の積算処理が作った `root_data/20261111_tracks/run*.root` には raw ツリーが無いため、入力にはできません。

ROOTファイルがまだ無い場合は、ターミナルで例えば次を実行します。
`python3` は EricaPy をビルドした Python、ノートブックは numpy / PyROOT のある「EricaNext」カーネルを使います。

```bash
source ~/opt/setup_ericanext.sh
cd ~/ANLNext/Erica_git_dev/EricaNext
mkdir -p tutorials/cache
python3 run_script/run_chain.py smile3_raw --compression 505 \
  -o tutorials/cache/tpc_raw_20261111_run71.root \
  --data-dir ~/QL_PC/storage_g0/20261111_run71 \
  --extra-data-dir ~/QL_PC/storage_g1/20261111_run71
```

`smile3_raw` は既定で `save_hits=true`, `save_fadc=true`, `save_threshold=0.0` です。
波形を閾値で間引いたファイルでは小さい信号が欠けるので、ゲイン較正には間引かないものを使います。

In [ ]:
RAWS = [CACHE / 'tpc_raw_20261111_run71.root']  # 入力。必要に応じてファイルを追加

In [ ]:
for path in RAWS:
    if not path.is_file(): raise FileNotFoundError(path)
    f = ROOT.TFile.Open(str(path))
    if not f or f.IsZombie(): raise RuntimeError(f'ROOTファイルを開けません: {path}')
    t = f.Get(TREE)
    if not t: raise RuntimeError(f'{path}: {TREE} がありません')
    required = ['hit_a_strip', 'hit_a_clock', 'hit_c_strip', 'hit_c_clock']
    missing = [b for b in required if not t.GetBranch(b)]
    if missing: raise RuntimeError(f'{path}: 必要なブランチがありません: {missing}')
    print(path.name, t.GetEntries(), 'events')
    f.Close()

## 2. 飛跡の積算図とクロック分布

全イベント・全ストリップのヒットを数えます（選別なし、TOTの各クロックを数える）。
RDataFrameでファイルごとに集計するため、大きなrawツリー全体をnumpyに展開しません。
色は対数、空ビンはマスク、最小値1です。ストリップ番号はボードIDに基づく物理番号（0〜1535）。
未使用ボードの領域も軸に残します。クロック分布はこの積算図の全ストリップ射影です。

In [ ]:
OUT = CACHE / 'tpc_drift'
OUT.mkdir(exist_ok=True)
NSTRIP = 1536  # 12 boards × 128 strips [strip]
NCLK = 1024   # エンコーダーの記録範囲 [clock]
images, clocks = {}, {}
for side in ('a', 'c'):
    total = None
    for i, path in enumerate(RAWS):
        df = ROOT.RDataFrame(TREE, str(path))
        h = df.Histo2D((f'image_{side}_{i}', '', NSTRIP, -0.5, NSTRIP-0.5,
                        NCLK, -0.5, NCLK-0.5), f'hit_{side}_strip', f'hit_{side}_clock').GetValue()
        if total is None:
            total = h.Clone(f'Image_{side.upper()}'); total.SetDirectory(0)
        else:
            total.Add(h)
    if total.GetEntries() <= 0: raise RuntimeError(f'{side}: ヒットがありません')
    if total.Integral() != total.GetEntries():
        raise RuntimeError('軸範囲外のヒットがあります。NSTRIP/NCLKを確認してください')
    images[side] = total
    clocks[side] = total.ProjectionY(f'clock_{side}', 1, NSTRIP)
    clocks[side].SetDirectory(0)
    print(side, int(total.GetEntries()), 'hits')

fig, axs = plt.subplots(1, 2, figsize=(16, 5))
for ax, (side, h) in zip(axs, images.items()):
    z = np.array([[h.GetBinContent(x+1,y+1) for x in range(NSTRIP)] for y in range(NCLK)])
    m = ax.imshow(np.ma.masked_less(z, 1), origin='lower', aspect='auto',
                  extent=(-0.5,NSTRIP-0.5,-0.5,NCLK-0.5),
                  norm=LogNorm(vmin=1, vmax=max(2, z.max())), interpolation='nearest')
    ax.set(title='Anode' if side == 'a' else 'Cathode', xlabel='Strip number', ylabel='Clock')
    fig.colorbar(m, ax=ax, label='Hit count')
fig.tight_layout(); fig.savefig(OUT / 'tracks.png', dpi=150); plt.show()
fout = ROOT.TFile(str(OUT / 'tracks.root'), 'RECREATE')
for h in [*images.values(), *clocks.values()]: h.Write()
fout.Close()

## 3. クロック分布と両端のフィット

まずクロック分布を見て、GEM側と天板側の端を探す範囲を設定します。
GEM側は隣接ビン差の絶対値が最大の位置、天板側はその差分の山をガウスでフィットします。
初期範囲は既存スクリプトの探索範囲です。速度から逆算して端を決めるものではありません。
鳴り続けるストリップ、端以外の急な構造、異なる条件の混在は積算図でも確認してください。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, (side, h) in zip(axs, clocks.items()):
    x = np.array([h.GetBinCenter(i) for i in range(1, NCLK+1)])
    y = np.array([h.GetBinContent(i) for i in range(1, NCLK+1)])
    ax.step(x, y, where='mid'); ax.set(title=side, xlabel='Clock', ylabel='Hit count')
fig.tight_layout(); plt.show()

In [ ]:
Z_GEM = 10.0          # μ-PIC〜GEM [mm]
DRIFT_LENGTH = 317.0  # μ-PIC〜天板 [mm]
GEM_RANGE = (0, 300)  # GEM側端の探索範囲 [clock]
TOP_RANGE = (400, 1023)  # 天板側端の探索・フィット範囲 [clock]
assert 0 <= GEM_RANGE[0] < GEM_RANGE[1] < TOP_RANGE[0] < TOP_RANGE[1] < NCLK
assert DRIFT_LENGTH > Z_GEM
from make_tpc_drift_velocity import analyze, qa_figure
results = {name: analyze(clocks[side], GEM_RANGE, TOP_RANGE, DRIFT_LENGTH-Z_GEM)
           for side, name in [('a', 'anode'), ('c', 'cathode')]}
qa_figure(str(OUT / 'drift_fit.png'), results)
display(Image(filename=str(OUT / 'drift_fit.png')))
for side, r in results.items():
    print(side, 'offset [clock]:', r['offset'], 'top [clock]:', r.get('edge'),
          'velocity [mm/clock]:', r.get('velocity'), 'NG:', r['problems'])

## 4. 較正ファイルを保存して使う

QAで端を確認してから次のセルを実行します。既存スクリプトが再計算し、
フィット不良・天板側の端が不明瞭・箱型でない等のチェックに失敗すればファイルを書きません。
`--force` は使いません。自動チェックだけで測定の妥当性が保証されるわけではありません。
アノードとカソードで求めたoffset・速度の算術平均を保存します。
出力はチュートリアル用cacheであり、`calib/tpc` を自動で上書きしません。

In [ ]:
CAL = OUT / 'drift_velocity.dat'
cmd = [PYTHON, str(SCRIPTS / 'make_tpc_drift_velocity.py'), '-o', str(CAL),
       *map(str, RAWS), '--z-gem', str(Z_GEM), '--drift-length', str(DRIFT_LENGTH),
       '--gem-range', *map(str, GEM_RANGE), '--top-range', *map(str, TOP_RANGE),
       '--nclk', str(NCLK), '--overwrite']
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print(CAL.read_text())

解析時に `--set TOT_Skewness.drift_velocity_file=<上のdrift_velocity.datの絶対パス>` を指定します。
`TOT_Skewness.z_gem_mm` も同じ `Z_GEM` にそろえてください。
較正ファイルの形式は `clock_offset <clock>` と `velocity <mm/clock>` です。